In [ ]:
"""
SLEEP STATE CLASSIFICATION - EARLY FUSION MODEL
================================================
Using pre-built Data_loader_new.py and model.py modules
"""

import pandas as pd
import numpy as np
import os
import tensorflow as tf
import json
from datetime import datetime
from collections import Counter
from sklearn.metrics import (classification_report, confusion_matrix, 
                             roc_curve, auc, f1_score)
import matplotlib.pyplot as plt
import seaborn as sns
import gc

# Import custom modules
from Data_loader_new import process_each_file_StackedModel, load_config
from model import build_stacked_model

# Set seeds
tf.random.set_seed(1)
np.random.seed(1)

print("TensorFlow Version:", tf.__version__)
print("Num GPUs Available:", len(tf.config.list_physical_devices('GPU')))

In [ ]:
# Load configuration
config = load_config()

# Training parameters
FOLD_NUM = 5
config['windowing']['window_size_seconds'] = 60
config['windowing']['step_size_seconds'] = 30
config['windowing']['batch_size'] = 32
config['segmentation']['segment_length_seconds'] = 30

batch_size = config['windowing']['batch_size']
window_size_seconds = config['windowing']['window_size_seconds']
step_size_seconds = config['windowing']['step_size_seconds']

print(f"Configuration loaded for Fold {FOLD_NUM}")
print(f"Window: {window_size_seconds}s, Step: {step_size_seconds}s, Batch: {batch_size}")

In [ ]:
# Define fold data paths
if FOLD_NUM == 1:
    train_dirs = ['../data/5folds_adhd/fold3/', '../data/5folds_adhd/fold2/', '../data/5folds_adhd/fold1/']
    val_dirs = ['../data/5folds_adhd/fold4/']
    test_dirs = ['../data/5folds_adhd/fold5/']
elif FOLD_NUM == 2:
    train_dirs = ['../data/5folds_adhd/fold4/', '../data/5folds_adhd/fold3/', '../data/5folds_adhd/fold2/']
    val_dirs = ['../data/5folds_adhd/fold5/']
    test_dirs = ['../data/5folds_adhd/fold1/']
elif FOLD_NUM == 3:
    train_dirs = ['../data/5folds_adhd/fold5/', '../data/5folds_adhd/fold4/', '../data/5folds_adhd/fold3/']
    val_dirs = ['../data/5folds_adhd/fold1/']
    test_dirs = ['../data/5folds_adhd/fold2/']
elif FOLD_NUM == 4:
    train_dirs = ['../data/5folds_adhd/fold1/', '../data/5folds_adhd/fold5/', '../data/5folds_adhd/fold4/']
    val_dirs = ['../data/5folds_adhd/fold2/']
    test_dirs = ['../data/5folds_adhd/fold3/']
elif FOLD_NUM == 5:
    train_dirs = ['../data/5folds_adhd/fold2/', '../data/5folds_adhd/fold1/', '../data/5folds_adhd/fold5/']
    val_dirs = ['../data/5folds_adhd/fold3/']
    test_dirs = ['../data/5folds_adhd/fold4/']

print(f"Train: {len(train_dirs)} folds, Val: {len(val_dirs)}, Test: {len(test_dirs)}")

In [ ]:
# Create datasets using Data_loader_new.py
train_dataset = tf.data.Dataset.from_generator(
    lambda: process_each_file_StackedModel(train_dirs, config),
    output_types=(tf.float32, tf.int64),
    output_shapes=((None, None, 4), ())
)

val_dataset = tf.data.Dataset.from_generator(
    lambda: process_each_file_StackedModel(val_dirs, config),
    output_types=(tf.float32, tf.int64),
    output_shapes=((None, None, 4), ())
)

# Optimize datasets
train_dataset = train_dataset.cache().shuffle(800).batch(batch_size).prefetch(tf.data.experimental.AUTOTUNE)
val_dataset = val_dataset.cache().shuffle(800).batch(batch_size).prefetch(tf.data.experimental.AUTOTUNE)

In [ ]:
# Calculate input shape
window_size_samples = int(window_size_seconds * config['frequencies']['ppg'])
input_shape = (window_size_samples, 3, 4)  # (1500, 3, 4) for 60s @ 25Hz

# Model hyperparameters
MODEL_CONFIG = {
    'lr': 0.0001,
    'num_filters_1': 16,
    'num_filters_2': 32,
    'num_filters_3': 64,
    'dropout_rate': 0.3,
    'dense_units': 128,
    'l2_regularization': 0.0001,
    'lstm_units': 128,
    'loss_gamma': 6,
    'loss_alpha': 0.2,
    'lr_alpha': 0.1,
    'lr_decay_steps': 12100
}

print(f"Input shape: {input_shape}")
print("\nModel Configuration:")
for key, value in MODEL_CONFIG.items():
    print(f"  {key}: {value}")

In [ ]:
# Build model using model.py // mentioned in seperated file 
model = build_stacked_model(input_shape, num_classes=2, model_config=MODEL_CONFIG)

# Display summary
model.summary()


print(f"\nTotal parameters: {model.count_params():,}")

In [ ]:
class GarbageCollectionCallback(tf.keras.callbacks.Callback):
    def on_epoch_end(self, epoch, logs=None):
        tf.keras.backend.clear_session()
        gc.collect()

# Define model save path
model_name = (f'fold{FOLD_NUM}_modelLstm128_earlyFusion_WithTemp_'
              f'removedNS_highpass01_order5_NoPeakRemoval_ppghighpass02low5_'
              f'StanScaleAll_BinFocalLossG6a02_lrdecayCos12ka01_'
              f'win{window_size_seconds}_step{step_size_seconds}_'
              f'batch{batch_size}_FilterSegment{config["segmentation"]["segment_length_seconds"]}_'
              f'lr{MODEL_CONFIG["lr"]}')

model_path = f'../models/fold_{FOLD_NUM}/{model_name}.h5'
log_dir = f'../logs/fit/{model_name}_{datetime.now().strftime("%Y%m%d-%H%M%S")}'

# Setup callbacks
checkpoint = tf.keras.callbacks.ModelCheckpoint(
    model_path, monitor='val_auc', save_best_only=True, mode='max', verbose=1
)

tensorboard = tf.keras.callbacks.TensorBoard(log_dir=log_dir, histogram_freq=1)

early_stopping = tf.keras.callbacks.EarlyStopping(
    monitor='val_loss', patience=20, restore_best_weights=True, verbose=1
)

garbage_collection = GarbageCollectionCallback()

callbacks = [checkpoint, tensorboard, early_stopping, garbage_collection]

print(f"Model will be saved to: {model_path}")

In [ ]:
print(f"\nStarting training for Fold {FOLD_NUM}...")

history = model.fit(
    train_dataset,
    epochs=100,
    validation_data=val_dataset,
    callbacks=callbacks,
    verbose=1
)

print("\nTraining completed")

### Testing Process 

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(15, 10))

# Loss
axes[0, 0].plot(history.history['loss'], label='Train Loss')
axes[0, 0].plot(history.history['val_loss'], label='Val Loss')
axes[0, 0].set_title('Model Loss')
axes[0, 0].set_xlabel('Epoch')
axes[0, 0].set_ylabel('Loss')
axes[0, 0].legend()
axes[0, 0].grid(True)

# AUC
axes[0, 1].plot(history.history['auc'], label='Train AUC')
axes[0, 1].plot(history.history['val_auc'], label='Val AUC')
axes[0, 1].set_title('Model AUC')
axes[0, 1].set_xlabel('Epoch')
axes[0, 1].set_ylabel('AUC')
axes[0, 1].legend()
axes[0, 1].grid(True)

# Sensitivity at Specificity
axes[1, 0].plot(history.history['sens@spec99'], label='Train')
axes[1, 0].plot(history.history['val_sens@spec99'], label='Val')
axes[1, 0].set_title('Sensitivity @ 99% Specificity')
axes[1, 0].set_xlabel('Epoch')
axes[1, 0].set_ylabel('Sensitivity')
axes[1, 0].legend()
axes[1, 0].grid(True)

# Specificity at Sensitivity
axes[1, 1].plot(history.history['spec@sens99'], label='Train')
axes[1, 1].plot(history.history['val_spec@sens99'], label='Val')
axes[1, 1].set_title('Specificity @ 99% Sensitivity')
axes[1, 1].set_xlabel('Epoch')
axes[1, 1].set_ylabel('Specificity')
axes[1, 1].legend()
axes[1, 1].grid(True)

plt.tight_layout()
plt.savefig(f'../results/fold_{FOLD_NUM}/{model_name}_training_history.png', dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
def plot_confusion_matrix(conf_matrix, labels, fname):
    plt.figure(figsize=(10, 7))
    sns.heatmap(conf_matrix, annot=True, fmt='d', cmap='Blues', 
                xticklabels=labels, yticklabels=labels)
    plt.xlabel('Predicted')
    plt.ylabel('True')
    plt.title('Confusion Matrix')
    plt.savefig(f"../results/fold_{FOLD_NUM}/{fname}_Conf_mat.png", dpi=300, bbox_inches='tight')
    plt.show()
    plt.close()

def plot_roc_curve(fpr, tpr, roc_auc, filename):
    plt.figure(figsize=(8, 6))
    plt.plot(fpr, tpr, color='darkorange', lw=2, label=f'ROC curve (area = {roc_auc:.2f})')
    plt.plot([0, 1], [0, 1], color='navy', lw=2, linestyle='--')
    plt.xlim([0.0, 1.0])
    plt.ylim([0.0, 1.05])
    plt.xlabel('False Positive Rate')
    plt.ylabel('True Positive Rate')
    plt.title('Receiver Operating Characteristic')
    plt.legend(loc='lower right')
    plt.grid(True, alpha=0.3)
    plt.savefig(filename, dpi=300, bbox_inches='tight')
    plt.show()
    plt.close()

def calculate_sensitivity_specificity_at_thresholds(fpr, tpr, thresholds, specificities):
    sensitivities = {}
    for spec in specificities:
        idx = np.where(fpr <= 1 - spec)[0]
        if len(idx) == 0:
            sensitivities[spec] = 0.0
        else:
            sensitivity = tpr[idx[-1]]
            sensitivities[spec] = sensitivity
            print(f"At {spec*100}% specificity, threshold: {thresholds[idx[-1]]:.4f}, sensitivity: {sensitivity:.4f}")
    return sensitivities

def advanced_smooth_predictions_sleep(pred_labels, pred_probs, optimal_threshold, 
                                     neighbor_window=10, margin=0.1):
    smoothed = pred_labels.copy()
    n = len(pred_labels)
    
    for i in range(n):
        start = max(0, i - neighbor_window)
        end = min(n, i + neighbor_window + 1)
        neighbor_labels = [pred_labels[j] for j in range(start, end) if j != i]
        neighbor_probs = [pred_probs[j] for j in range(start, end) if j != i]
        
        if len(neighbor_labels) == 0:
            continue
        
        majority_label = 1 if sum(neighbor_labels) > len(neighbor_labels) / 2 else 0
        avg_neighbor_prob = np.mean(neighbor_probs)
        
        if pred_labels[i] == 0 and majority_label == 1 and pred_probs[i] > (optimal_threshold - margin):
            smoothed[i] = 1
    
    return smoothed

print("Evaluation functions defined")

In [ ]:
# Update config for testing (non-overlapping windows)
config['windowing']['step_size_seconds'] = 60
test_batch_size = 64

# Create test dataset
test_dataset = tf.data.Dataset.from_generator(
    lambda: process_each_file_StackedModel(test_dirs, config),
    output_types=(tf.float32, tf.int64),
    output_shapes=((None, None, 4), ())
)

test_dataset = test_dataset.batch(test_batch_size).prefetch(tf.data.experimental.AUTOTUNE)

print(f"Test dataset created for Fold {FOLD_NUM}")

In [ ]:
# Load best model
print(f"Loading model from: {model_path}")
model = tf.keras.models.load_model(model_path)

# Generate predictions
y_true = []
y_pred_prob = []

print("Generating predictions on test set...")
for batch in test_dataset:
    x, y = batch
    preds = model.predict(x, verbose=0)
    y_true.extend(y.numpy())
    y_pred_prob.extend(preds)

y_true = np.array(y_true)
y_pred_prob = np.array(y_pred_prob).reshape(-1)

label_counts = Counter(y_true)
print(f"\nLabel distribution: {label_counts}")
print(f"Total predictions: {len(y_true)}")

In [ ]:
# Calculate ROC curve
fpr, tpr, thresholds = roc_curve(y_true, y_pred_prob)
roc_auc = auc(fpr, tpr)

# Find optimal threshold using Youden's J statistic
optimal_idx = np.argmax(tpr - fpr)
optimal_threshold = thresholds[optimal_idx]

print(f"ROC AUC: {roc_auc:.4f}")
print(f"Optimal Threshold: {optimal_threshold:.4f}")

# Plot ROC curve
plot_roc_curve(fpr, tpr, roc_auc, 
               filename=f'../results/fold_{FOLD_NUM}/{model_name}_roc_curve.png')

# Sensitivity at different specificity levels
specificities = [0.6, 0.70, 0.80, 0.85, 0.90]
sensitivities = calculate_sensitivity_specificity_at_thresholds(fpr, tpr, thresholds, specificities)

print("\nSensitivity at different specificity levels:")
for spec, sens in sensitivities.items():
    print(f"  {int(spec * 100)}% Specificity: {sens:.4f}")

In [ ]:
# Confusion matrix
conf_matrix = confusion_matrix(y_true, y_pred)
class_report = classification_report(y_true, y_pred, target_names=['Awake', 'Sleep'])

tn, fp, fn, tp = conf_matrix.ravel()
sensitivity = tp / (tp + fn)
specificity = tn / (tn + fp)
f1 = f1_score(y_true, y_pred)

print("\n" + "="*60)
print(f"FOLD {FOLD_NUM} TEST RESULTS")
print("="*60)
print(f"ROC AUC: {roc_auc:.4f}")
print(f"F1 Score: {f1:.4f}")
print(f"Sensitivity: {sensitivity:.4f}")
print(f"Specificity: {specificity:.4f}")
print(f"\nConfusion Matrix:")
print(f"  TN: {tn}, FP: {fp}")
print(f"  FN: {fn}, TP: {tp}")
print(f"\nClassification Report:")
print(class_report)

# Plot confusion matrix
plot_confusion_matrix(conf_matrix, labels=['Awake', 'Sleep'], fname=model_name)

In [ ]:
# Save comprehensive results
with open(f'../results/fold_{FOLD_NUM}/{model_name}_classification_report.txt', 'w') as f:
    f.write(f"Label Counts: {label_counts}\n\n")
    f.write(f"ROC AUC: {roc_auc:.4f}\n\n")
    f.write("Sensitivity at different specificities:\n")
    for spec, sens in sensitivities.items():
        f.write(f"Sensitivity at {int(spec * 100)}% Specificity: {sens:.4f}\n")
    f.write(f"\nOptimal Threshold (Youden's J): {optimal_threshold:.4f}\n\n")
    f.write(f"F1 Score: {f1:.4f}\n\n")
    f.write("Classification Report:\n")
    f.write(class_report + "\n")
    f.write(f"Confusion Matrix:\n")
    conf_matrix_str = "\n".join(['\t'.join([str(cell) for cell in row]) for row in conf_matrix])
    f.write(conf_matrix_str + "\n\n")
    f.write(f"Sensitivity: {sensitivity:.4f}\n")
    f.write(f"Specificity: {specificity:.4f}\n")

print(f"\nResults saved to: ../results/fold_{FOLD_NUM}/{model_name}_classification_report.txt")